In [ ]:
from pathlib import Path
import re

import pandas as pd
import mercury as mr


LANGUAGE_PATTERNS = {
    "Python": r"python",
    "JavaScript / TypeScript": r"javascript|typescript|node\.?js",
    "Java": r"\bjava\b",
    "Go": r"\bgolang\b|\bgo (?:developer|engineer|programming|backend)",
    "Rust": r"\brust\b",
    "C# / .NET": r"c#|\.net",
    "C++": r"c\+\+",
    "Ruby": r"\bruby\b",
    "PHP": r"\bphp\b",
    "Kotlin": r"\bkotlin\b",
    "Swift": r"\bswift\b",
    "Scala": r"\bscala\b",
    "R": r"\br language\b|\brstudio\b|\btidyverse\b",
    "SQL": r"\bsql\b",
}

ROLE_PATTERNS = {
    "All": None,
    "Data / ML": (
        r"\bdata science\b|\bdata scientist(?:s)?\b|"
        r"\bdata engineer(?:s|ing)?\b|\bmachine learning\b|"
        r"\bml\b|\bmlops\b"
    ),
    "Backend": r"\bbackend\b|\bback-end\b|\bserver-side\b|\bapi engineer",
    "Frontend": r"\bfrontend\b|\bfront-end\b|\breact\b|\bvue\b|\bangular\b",
    "DevOps / Cloud": (
        r"\bdevops\b|\bsre\b|\bsite reliability\b|"
        r"\bplatform engineer|\bcloud engineer|\bkubernetes\b"
    ),
    "Security": r"\bsecurity\b|\bcybersecurity\b|\binfosec\b|\bapplication security\b",
}

LOCATION_PATTERNS = {
    "All": None,
    "Remote": r"remote",
    "Hybrid": r"\bhybrid\b",
    "On-site / office": (
        r"\bon[- ]?site\b|\bin[- ]?office\b|\boffice[- ]based\b|"
        r"\bwork(?:ing)? from (?:our|the) office\b"
    ),
}

LOCATION_TITLE_LABELS = {
    "All": "",
    "Remote": "remote",
    "Hybrid": "hybrid",
    "On-site / office": "on-site",
}

ROLE_TITLE_LABELS = {
    "All": "",
    "Data / ML": "data/ML",
    "Backend": "backend",
    "Frontend": "frontend",
    "DevOps / Cloud": "DevOps/cloud",
    "Security": "security",
}

COMPENSATION_PATTERN = r"[$€£]\s?\d|\bcompensation\b|\bcomp\b"

data_roots = [Path.cwd(), Path.cwd() / "who-is-hiring"]
data_dir = next(
    (root for root in data_roots if list(root.glob("who_is_hiring_[0-9][0-9][0-9][0-9].csv.gz"))),
    None,
)
if data_dir is None:
    raise FileNotFoundError("Could not find the yearly Who is Hiring CSV files.")

year_files = {
    int(match.group(1)): path
    for path in data_dir.glob("who_is_hiring_*.csv.gz")
    if (match := re.fullmatch(r"who_is_hiring_(\d{4})\.csv\.gz", path.name))
}
available_years = sorted(year_files, reverse=True)

In [ ]:
default_year = "2025" if 2025 in year_files else str(available_years[0])
year_filter = mr.Select(
    label="Year",
    value=default_year,
    choices=[str(year) for year in available_years],
    url_key="year",
    key="year-filter",
)

In [ ]:
selected_year = int(year_filter.value)
jobs = pd.read_csv(year_files[selected_year])
jobs["search_text"] = jobs["comment"].fillna("").astype(str)
jobs["thread_timestamp"] = pd.to_datetime(jobs["thread_timestamp"], utc=True)

In [ ]:
location_filter = mr.Select(
    label="Work arrangement",
    value="Remote",
    choices=list(LOCATION_PATTERNS),
    url_key="location",
    key="location-filter",
)

In [ ]:
language_filter = mr.Select(
    label="Programming language",
    value="Python",
    choices=list(LANGUAGE_PATTERNS),
    url_key="language",
    key="language-filter",
)

In [ ]:
role_filter = mr.Select(
    label="Role family",
    value="Data / ML",
    choices=list(ROLE_PATTERNS),
    url_key="role",
    key="role-filter",
)

In [ ]:
job_description = " ".join(
    part
    for part in [
        LOCATION_TITLE_LABELS[location_filter.value],
        language_filter.value,
        ROLE_TITLE_LABELS[role_filter.value],
    ]
    if part
)
article = "an" if job_description[0].lower() in "aeiou" else "a"
_ = mr.Markdown(
    f"# How hard is it to find {article} {job_description} job?\n\n"
    "Start with every Ask HN ‘Who is hiring?’ entry in the selected year, "
    "then progressively apply transparent work-arrangement, programming-language, "
    "role, and compensation filters.",
    key="dynamic-title",
)

In [ ]:
percentage_filter = mr.Select(
    label="Percentage comparison",
    value="From all entries",
    choices=["From all entries", "From previous stage"],
    url_key="compare",
    key="percentage-filter",
)

In [ ]:
text = jobs["search_text"]
language_mask = text.str.contains(
    LANGUAGE_PATTERNS[language_filter.value], case=False, regex=True, na=False
)
compensation_mask = text.str.contains(
    COMPENSATION_PATTERN, case=False, regex=True, na=False
)

stage_masks = [pd.Series(True, index=jobs.index)]
stage_names = [f"All {selected_year} entries"]

location_pattern = LOCATION_PATTERNS[location_filter.value]
if location_pattern is not None:
    location_mask = text.str.contains(
        location_pattern, case=False, regex=True, na=False
    )
    stage_masks.append(stage_masks[-1] & location_mask)
    stage_names.append(f"Mention {location_filter.value.lower()}")

stage_masks.append(stage_masks[-1] & language_mask)
stage_names.append(f"+ {language_filter.value}")
role_pattern = ROLE_PATTERNS[role_filter.value]
if role_pattern is not None:
    role_mask = text.str.contains(
        role_pattern, case=False, regex=True, na=False
    )
    stage_masks.append(stage_masks[-1] & role_mask)
    stage_names.append(f"+ {role_filter.value}")
stage_masks.append(stage_masks[-1] & compensation_mask)
stage_names.append("+ salary / compensation")
stage_values = [int(mask.sum()) for mask in stage_masks]
funnel_data = pd.DataFrame({"stage": stage_names, "entries": stage_values})
matches = jobs[stage_masks[-1]].copy()

In [ ]:
overall_rate = stage_values[-1] / stage_values[0] if stage_values[0] else 0
filter_rates = [
    1 - (after / before) if before else 0
    for before, after in zip(stage_values, stage_values[1:])
]
most_selective_index = max(range(len(filter_rates)), key=filter_rates.__getitem__)
most_selective_name = stage_names[most_selective_index + 1].removeprefix("+ ")

mr.Indicator([
    mr.Indicator(f"{stage_values[0]:,}", label="Starting entries"),
    mr.Indicator(f"{stage_values[-1]:,}", label="Final matches"),
    mr.Indicator(f"{overall_rate:.1%}", label="Overall match rate"),
    mr.Indicator(
        f"{filter_rates[most_selective_index]:.1%} filtered out",
        label=f"Most selective filter: {most_selective_name}",
    ),
])

## Opportunity funnel

In [ ]:
mr.Funnel(
    funnel_data,
    stage="stage",
    value="entries",
    percentage=(
        "first" if percentage_filter.value == "From all entries" else "previous"
    ),
    height=430,
    show_values=True,
    show_percentage=True,
)

In [ ]:
location_description = (
    "any work arrangement"
    if location_filter.value == "All"
    else f"{location_filter.value.lower()} work"
)
_ = mr.Markdown(
    f"### The takeaway\n\n"
    f"The transparent keyword search narrows **{stage_values[0]:,}** {selected_year} entries "
    f"to **{stage_values[-1]:,}** matching {location_description}, "
    f"{language_filter.value}, "
    f"{('any role family' if role_filter.value == 'All' else role_filter.value)}, "
    f"and a recognizable compensation signal—"
    f"**{overall_rate:.1%}** of the starting set.",
    key="takeaway",
)

## Matching entries

These are keyword matches, not independently verified job requirements. Use the Hacker News URL to read each original comment.

In [ ]:
match_table = pd.DataFrame({
    "Month": matches["thread_timestamp"].dt.strftime("%B"),
    "Author": matches["author"].fillna("Unknown"),
    "Comment preview": (
        matches["comment"].fillna("").str.replace(r"\s+", " ", regex=True).str.slice(0, 280)
    ),
    "Hacker News URL": (
        "https://news.ycombinator.com/item?id=" + matches["comment_id"].astype(str)
    ),
})

_ = mr.Table(
    match_table,
    page_size=20,
    search=True,
    height="520px",
    key="matching-entries",
)

In [ ]:
_ = mr.Markdown(
    f"""## Methodology and limitations

Each CSV row is treated as one opportunity entry. A comment can advertise multiple jobs, and keyword mentions do not guarantee that a technology is mandatory. “Remote” can include geographically restricted remote work.

Matching is case-insensitive and cumulative. The active work arrangement is **{location_filter.value}**, the active language is **{language_filter.value}**, and the active role family is **{role_filter.value}**. Work-arrangement categories can overlap because a posting may mention more than one option. Selecting **All** for work arrangement or role family omits that stage from the funnel. The final stage looks for `compensation`, standalone `comp`, or a currency symbol (`$`, `€`, or `£`) followed by a number. Counts can differ from other analyses that use broader or narrower keyword definitions.

The local dataset contains comments from monthly [Ask HN: Who is hiring?](https://news.ycombinator.com/) threads. Every displayed result links back to its original Hacker News comment.

Built with [Mercury's Funnel widget](https://runmercury.com/docs/output/funnel/).""",
    key="methodology",
)